# Decision tree

Seed 42, hyperparameters chosen by a TPE search and features chosen by
out-of-fold permutation importance — neither by hand. Fitted and scored below,
then pointed at 2026, a season nobody has played yet.

Every model studied gets the same three cells: this heading, the fit, and the
2026 prediction. The harness the later ones reuse — the folds, the search, the
permutation table, the 2026 forecast — is written in this first block and takes
the estimator as an argument, so a second model is a class and a search space.
2026 is what will say which of them was right.

In [1]:
import warnings

import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from nfl_trees import metrics as metrics_mod
from nfl_trees.config import FeatureConfig
from nfl_trees.data import load_scores
from nfl_trees.features import build_dataset, make_preprocessor

SEED = 42
N_TRIALS = 80
HOLDOUT = 2025
METRICS = ["roc_auc", "accuracy", "log_loss", "brier"]

# The ten candidate features of notebook `02`, by the builder that makes them.
CANDIDATES = {
    "calendar": ["month", "week", "day", "playoff"],
    "win_rates": ["pct_home_win", "pct_away_win"],
    "drive_rates": [
        "home_pct_score_drive",
        "home_pct_allowed_drive",
        "away_pct_score_drive",
        "away_pct_allowed_drive",
    ],
}
CATEGORICAL = {"day"}
ALL_COLUMNS = [c for cols in CANDIDATES.values() for c in cols]


def config_for(columns):
    """A `FeatureConfig` over `columns`, dropping builders nothing survives from."""
    columns = [c for c in ALL_COLUMNS if c in set(columns)]
    return FeatureConfig(
        numeric=[c for c in columns if c not in CATEGORICAL],
        categorical=[c for c in columns if c in CATEGORICAL],
        builders=[b for b, cols in CANDIDATES.items() if set(cols) & set(columns)],
    )


# `drive_rates` folds every plays file, so the first run takes a few seconds.
FULL = config_for(ALL_COLUMNS)
games = load_scores()
X, y, meta = build_dataset(games, FULL, "home_win")
y = y.astype(int)
season = meta["Season"].astype(int)

# 2010 is the warm-up season the history rates have no previous season for, and
# the holdout is walled off from everything below: neither the hyperparameter
# search nor the feature selection is allowed to see 2025.
TUNING_SEASONS = [s for s in sorted(season.unique()) if 2013 <= s < HOLDOUT]


# --------------------------------------------------------------------------- #
# the harness, shared by every model in this notebook
# --------------------------------------------------------------------------- #
# Two things change from one model to the next: the estimator class and the
# space its hyperparameters live in. Both arrive as arguments, so the folds, the
# search, the permutation table and the holdout report are written once here.


def fit_model(estimator, train, params, features):
    """The pipeline a config would produce: repo preprocessor, then the model."""
    pipe = Pipeline(
        [
            ("prep", make_preprocessor(features)),
            ("model", estimator(random_state=SEED, **params)),
        ]
    )
    return pipe.fit(X.loc[train, features.columns], y[train])


def over_folds(work, backend):
    """Run `work(season)` once per tuning season, in parallel.

    Twelve folds are twelve independent fits, and that is the level worth
    parallelizing -- but not with the same machinery for every model, which is
    why `backend` is an argument. Measured on these folds: a tree fits in
    milliseconds, so shipping a fold to another process costs more than the fold
    itself (2.7s against 0.7s in series) and threads are the answer; a forest of
    a few hundred trees fits in seconds and holds the GIL while it does, so
    threads buy almost nothing there and processes cut a trial from 5.5s to 2.1s.

    The model's own `n_jobs` stays at 1 either way. Two levels of parallelism
    over the same twelve cores only take turns.
    """
    return Parallel(n_jobs=-1, prefer=backend)(delayed(work)(s) for s in TUNING_SEASONS)


def cv_auc(estimator, params, features, backend="threads"):
    """Rolling-origin CV: each season scored by a model trained only on earlier ones.

    A single split would hand the search 280 games of noise to chase. Twelve
    seasons scored in sequence is the same discipline the season split enforces
    for a real run, applied twelve times.
    """

    def score(s):
        model = fit_model(estimator, (season >= 2011) & (season < s), params, features)
        test = season == s
        return metrics_mod.compute(
            "classification",
            ["roc_auc"],
            y[test].to_numpy(),
            model.predict(X.loc[test, features.columns]),
            model.predict_proba(X.loc[test, features.columns])[:, 1],
        )["roc_auc"]

    return float(np.mean(over_folds(score, backend)))


def tune(estimator, space, features, *, start_from=None, backend="threads", n_trials=N_TRIALS):
    """TPE over `space`, the knobs of this particular model, scored by `cv_auc`.

    `start_from` seeds the study with a known-good point, so a second search
    cannot end up worse than the first one it is meant to improve on. `n_trials`
    is per model rather than global: two budgets are only comparable when a trial
    costs something comparable, and here one costs milliseconds and the other
    seconds.

    Hands back the study as well, so a block can ask what the trials it did not
    win with were worth.
    """

    def objective(trial):
        return cv_auc(estimator, space(trial), features, backend)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        study = optuna.create_study(
            direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        if start_from is not None:
            study.enqueue_trial(start_from)
        study.optimize(objective, n_trials=n_trials)
    return study.best_params, study.best_value, study


def permutation_table(estimator, params, features, n_repeats=10, backend="threads"):
    """How much roc_auc each column is worth, measured where it has to be.

    Shuffling a column breaks its link to the outcome; whatever the model loses
    was what the column was carrying. Measured on the *held-out* season of every
    fold, never on the training rows: on the training set a model can look like
    it depends on a column it only memorised.
    """

    def shuffle(s):
        model = fit_model(estimator, (season >= 2011) & (season < s), params, features)
        test = season == s
        result = permutation_importance(
            model,
            X.loc[test, features.columns],
            y[test],
            scoring="roc_auc",
            n_repeats=n_repeats,
            random_state=SEED,
        )
        return pd.Series(result.importances_mean, index=features.columns)

    folds = pd.DataFrame(over_folds(shuffle, backend), index=TUNING_SEASONS)
    table = pd.DataFrame(
        {
            "mean": folds.mean(),
            "std_err": folds.std(ddof=1) / np.sqrt(len(folds)),
            "seasons_up": (folds > 0).sum(),
        }
    )
    table["t"] = table["mean"] / table["std_err"].replace(0.0, np.nan)
    return table.sort_values("mean", ascending=False)


def selected(importance):
    """Split a permutation table into the columns to keep and the ones to drop.

    A feature stays when its mean clears its own standard error *and* more than
    one season carried it. Two conditions rather than one because of `week`:
    positive in a single season out of twelve and exactly zero in the other
    eleven, which makes its mean and its standard error algebraically the same
    number -- `mean > 0` and `mean > std_err` would both come down to the last
    bit of a float rather than to anything about football.

    Cross-validation is deliberately not the judge here. Refitting without a
    column moves the model even when that column was never split on, because
    ties between equally good splits break differently, and that wobble is worth
    more roc_auc than any of these features: a CV-guided elimination on these
    folds drops `pct_home_win` (t = 2.6) and keeps `day` (importance exactly
    zero). Permutation importance is read off one fitted model per fold, so it
    never refits and never sees that noise. It decides; CV only checks the bill.
    """
    survives = (importance["mean"] > importance["std_err"]) & (importance["seasons_up"] > 1)
    return list(importance.index[survives]), list(importance.index[~survives])


def holdout_report(estimator, runs):
    """Fit each `(label, params, features, cv)` on 2011-2024 and score it on the holdout."""
    train, test = season.between(2011, 2024), season.eq(HOLDOUT)
    rows = {}
    for label, params, features, cv in runs:
        model = fit_model(estimator, train, params, features)
        rows[label] = {
            "cv_roc_auc": round(cv, 4),
            **{
                k: round(v, 3)
                for k, v in metrics_mod.compute(
                    "classification",
                    METRICS,
                    y[test].to_numpy(),
                    model.predict(X.loc[test, features.columns]),
                    model.predict_proba(X.loc[test, features.columns])[:, 1],
                ).items()
            },
        }
    return pd.DataFrame(rows).T


# --------------------------------------------------------------------------- #
# the model this block is about
# --------------------------------------------------------------------------- #
def tree_space(trial):
    """Five knobs, all bounded away from the pathological ends.

    `min_samples_leaf` starts at 10 because a leaf holding a handful of games
    returns probabilities of 0 or 1, and `log_loss` punishes every one of those
    that misses. `ccp_alpha` is cost-complexity pruning: the search can prune a
    deep tree back instead of only refusing to grow it.
    """
    return dict(
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        max_depth=trial.suggest_int("max_depth", 2, 12),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 250, log=True),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 200, log=True),
        ccp_alpha=trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
    )


# -- 1. tune on every candidate feature -------------------------------------- #
params_full, cv_full, _ = tune(DecisionTreeClassifier, tree_space, FULL)
print(f"TPE search, {N_TRIALS} trials over {len(TUNING_SEASONS)} folds (2013-2024)")
print("params  : " + "  ".join(f"{k}={v}" for k, v in params_full.items()))
print(f"cv auc  : {cv_full:.4f}   on all {len(FULL.columns)} features")

# -- 2. ask each feature what it is worth ------------------------------------ #
tree_importance = permutation_table(DecisionTreeClassifier, params_full, FULL)
print("\npermutation importance, out of fold: roc_auc lost when the column is shuffled")
print(tree_importance.round(5).to_string())

TREE_KEEP, TREE_DROPPED = selected(tree_importance)
print(f"\nkept    : {', '.join(TREE_KEEP)}")
print(f"dropped : {', '.join(TREE_DROPPED)}")

# -- 3. retune on the survivors ---------------------------------------------- #
TREE_FEATURES = config_for(TREE_KEEP)
TREE_PARAMS, cv_reduced, _ = tune(
    DecisionTreeClassifier, tree_space, TREE_FEATURES, start_from=params_full
)
cv_cut = cv_auc(DecisionTreeClassifier, params_full, TREE_FEATURES)
print(f"\nretuned on the {len(TREE_KEEP)} survivors, builders now {TREE_FEATURES.builders}")
print("params  : " + "  ".join(f"{k}={v}" for k, v in TREE_PARAMS.items()))
print(f"cv auc  : {cv_reduced:.4f}   ({cv_reduced - cv_full:+.4f} against all {len(FULL.columns)})")
print(f"the cut : {cv_cut:.4f} with the step 1 params ({cv_cut - cv_full:+.4f}) -- what the four")
print("          dropped columns were worth to a tree that never split on them")

# -- 4. the holdout neither step was allowed to see -------------------------- #
train, test = season.between(2011, 2024), season.eq(HOLDOUT)
tree_report = holdout_report(
    DecisionTreeClassifier,
    [
        (f"all {len(FULL.columns)}", params_full, FULL, cv_full),
        (f"kept {len(TREE_KEEP)}", TREE_PARAMS, TREE_FEATURES, cv_reduced),
    ],
)
print(f"\ntrain 2011-2024 ({int(train.sum())} games), holdout {HOLDOUT} ({int(test.sum())} games)")
print(tree_report.to_string())
print(f"always-home : accuracy {y[test].mean():.3f}   <- the bar to clear")

# What the next cell predicts with: the kept features, refit on every game played.
tree = fit_model(DecisionTreeClassifier, season.between(2011, HOLDOUT), TREE_PARAMS, TREE_FEATURES)
print(f"\nrefit on 2011-{HOLDOUT} ({int(season.between(2011, HOLDOUT).sum())} games) to predict 2026")

TPE search, 80 trials over 12 folds (2013-2024)
params  : criterion=log_loss  max_depth=5  min_samples_leaf=98  min_samples_split=37  ccp_alpha=2.235791197815289e-06
cv auc  : 0.6148   on all 10 features

permutation importance, out of fold: roc_auc lost when the column is shuffled
                           mean  std_err  seasons_up        t
away_pct_score_drive    0.05277  0.00770          12  6.85530
home_pct_allowed_drive  0.02285  0.00928           9  2.46233
home_pct_score_drive    0.02034  0.00647          10  3.14523
pct_home_win            0.01026  0.00396           7  2.59198
away_pct_allowed_drive  0.00757  0.00344           9  2.20309
pct_away_win            0.00236  0.00174           3  1.35331
week                    0.00013  0.00013           1  1.00000
month                   0.00000  0.00000           0      NaN
playoff                 0.00000  0.00000           0      NaN
day                     0.00000  0.00000           0      NaN

kept    : away_pct_score_drive, ho

In [2]:
from nfl_trees.data import canonical_team
from nfl_trees.features import apply_builders

DIVISIONS = {
    "AFC East": ["BUF", "MIA", "NE", "NYJ"],
    "AFC North": ["BAL", "CIN", "CLE", "PIT"],
    "AFC South": ["HOU", "IND", "JAX", "TEN"],
    "AFC West": ["DEN", "KC", "LAC", "LV"],
    "NFC East": ["DAL", "NYG", "PHI", "WAS"],
    "NFC North": ["CHI", "DET", "GB", "MIN"],
    "NFC South": ["ATL", "CAR", "NO", "TB"],
    "NFC West": ["ARI", "LAR", "SEA", "SF"],
}
DIVISION = {team: div for div, teams in DIVISIONS.items() for team in teams}
TEAMS = sorted(DIVISION)

ROUNDS = {
    "wild_card": ("WILD CARD WEEKEND", "January 10th"),
    "divisional": ("DIVISIONAL PLAYOFFS", "January 17th"),
    "championship": ("CONFERENCE CHAMPIONSHIPS", "January 24th"),
    "super_bowl": ("SUPER BOWL", "February 7th"),
}


def home_win_proba(model, features):
    """Build `frame -> P(the home team wins)`, one probability per row.

    `build_dataset` cannot be used here: it drops every row with a missing
    target, which is all of them when the games have not been played. The
    builders still work -- for any 2026 week the history rates read 2025 in
    full, which is exactly what a forecast made today has to go on.

    The model arrives as an argument rather than as a global so that every block
    of this notebook can run the same 2026 through its own.
    """

    def predict(frame):
        design = apply_builders(frame, features.builders)[features.columns]
        numeric = design[features.numeric].apply(pd.to_numeric, errors="coerce").astype(float)
        categorical = design[features.categorical].astype(object).where(
            design[features.categorical].notna(), np.nan
        )
        design = pd.concat([numeric, categorical], axis=1)[features.columns]
        return model.predict_proba(design)[:, 1]

    return predict


def regular_season(predict):
    """Pick all 272 games of 2026 and print the standings they add up to."""
    schedule = load_scores([2026], statuses=("TBD",), include_postseason=False)
    schedule = schedule.assign(
        home=canonical_team(schedule["HomeTeam"]), away=canonical_team(schedule["AwayTeam"])
    )
    schedule["p_home"] = predict(schedule)
    schedule["winner"] = np.where(schedule["p_home"] >= 0.5, schedule["home"], schedule["away"])

    # Two readings of the same 272 probabilities. `W-L` is the record the picks
    # add up to, and a team favoured every single week goes 17-0 in it; `exp`
    # sums the probabilities instead, which is the win total the model would
    # actually bet on.
    wins = schedule["winner"].value_counts().reindex(TEAMS).fillna(0)
    played = (
        schedule["home"].value_counts()
        .add(schedule["away"].value_counts(), fill_value=0)
        .reindex(TEAMS)
    )
    expected = (
        schedule.groupby("home")["p_home"].sum()
        .add(schedule.assign(p=1 - schedule["p_home"]).groupby("away")["p"].sum(), fill_value=0)
        .reindex(TEAMS)
    )

    standings = pd.DataFrame(
        {
            "division": [DIVISION[team] for team in TEAMS],
            "W": wins.astype(int),
            "L": (played - wins).astype(int),
            "exp": expected.round(1),
        },
        index=TEAMS,
    ).sort_values(["W", "exp"], ascending=False)
    standings["record"] = standings["W"].astype(str) + "-" + standings["L"].astype(str)

    print(f"2026 regular season, {len(schedule)} games predicted")
    print("W-L is the record the picks add up to, exp is the summed probabilities\n")
    for division in DIVISIONS:
        block = standings[standings["division"] == division]
        print(
            f"{division:<10}  "
            + "   ".join(f"{t:<3} {r.record:>5} ({r.exp:>4.1f})" for t, r in block.iterrows())
        )
    return standings


def seeds_of(standings, conference):
    """Seeds 1-7: the four division winners by record, then the three best left."""
    table = standings[standings["division"].str.startswith(conference)]
    champions = [block.index[0] for _, block in table.groupby("division", sort=False)]
    won_division = table.index.isin(champions)
    ranked = list(table.index[won_division]) + list(table.index[~won_division][:3])
    return list(enumerate(ranked, start=1))


def round_frame(pairs, round_key):
    week, date = ROUNDS[round_key]
    return pd.DataFrame(
        [
            {
                "Season": 2026, "Week": week, "GameStatus": "TBD", "GameSlot": "Sunday",
                "GameDate": date, "AwayTeam": away, "AwayScore": np.nan,
                "HomeTeam": home, "HomeScore": np.nan,
            }
            for (_, home), (_, away) in pairs
        ]
    )


def play_round(predict, standings, pairs, round_key, label, neutral=False):
    """Run one round. `pairs` is (host, visitor) as (seed, team); returns the winners.

    On a neutral field nobody hosts, and the model has no way to be told that --
    the four venue features are half of what it reads. So the Super Bowl is
    predicted twice, once with each team as the home side, and the two averaged.
    """
    p_first = predict(round_frame(pairs, round_key))
    if neutral:
        flipped = predict(round_frame([(b, a) for a, b in pairs], round_key))
        p_first = (p_first + (1 - flipped)) / 2

    print(f"\n{label}")
    winners = []
    for (first, second), p in zip(pairs, p_first):
        if abs(p - 0.5) < 1e-9:
            # The two sides came out at exactly 0.5: for a tree that means both
            # landed in the same leaf, for a forest that its trees split evenly.
            # Either way the model has nothing left to say, so the pair is decided
            # on the regular season it just predicted rather than on the order the
            # pair happens to be in.
            winner = max((first, second), key=lambda seed: standings.loc[seed[1], "exp"])
            note = "   (tied: settled on expected wins)"
        else:
            winner, note = (first if p > 0.5 else second), ""
        prob = p if winner == first else 1 - p
        print(
            f"  ({second[0]}) {second[1]:<3} {'vs' if neutral else 'at'} ({first[0]}) {first[1]:<3}"
            f"   ->  {winner[1]:<3} {prob:6.1%}{note}"
        )
        winners.append(winner)
    return winners


def playoffs(predict, standings):
    """Seed both conferences off the predicted standings and play the bracket out."""
    print("\n\n2026 playoffs, predicted -- the better seed hosts every round")
    finalists = {}
    for conference in ("AFC", "NFC"):
        seeds = seeds_of(standings, conference)
        print("\n" + conference + " seeds: " + "  ".join(f"{n}.{team}" for n, team in seeds))

        # Seed 1 sits out the wild card round; every later round re-seeds, so the
        # best team left always hosts the worst one left.
        alive = sorted(
            [seeds[0]]
            + play_round(
                predict,
                standings,
                [(seeds[1], seeds[6]), (seeds[2], seeds[5]), (seeds[3], seeds[4])],
                "wild_card",
                f"{conference} wild card   ({seeds[0][1]} on a bye)",
            )
        )
        alive = sorted(
            play_round(
                predict,
                standings,
                [(alive[0], alive[3]), (alive[1], alive[2])],
                "divisional",
                f"{conference} divisional",
            )
        )
        finalists[conference] = play_round(
            predict, standings, [(alive[0], alive[1])], "championship", f"{conference} championship"
        )[0]

    champion = play_round(
        predict,
        standings,
        [(finalists["AFC"], finalists["NFC"])],
        "super_bowl",
        "Super Bowl   (neutral field: predicted from both sides and averaged)",
        neutral=True,
    )[0]
    print(f"\n  champion: {champion[1]}")
    return champion


def forecast_2026(model, features):
    """The whole 2026 season as this model sees it: records, bracket, champion."""
    predict = home_win_proba(model, features)
    standings = regular_season(predict)
    playoffs(predict, standings)
    return standings


tree_standings = forecast_2026(tree, TREE_FEATURES)

2026 regular season, 272 games predicted
W-L is the record the picks add up to, exp is the summed probabilities

AFC East    NE   15-2 (11.2)   BUF   8-9 ( 8.8)   MIA  3-14 ( 7.9)   NYJ  1-16 ( 5.6)
AFC North   CIN  12-5 ( 9.5)   BAL  10-7 ( 9.3)   PIT  5-12 ( 7.9)   CLE  4-13 ( 5.6)
AFC South   HOU  16-1 (10.2)   JAX  15-2 (10.5)   IND  10-7 ( 9.6)   TEN  1-16 ( 4.6)
AFC West    DEN  14-3 (10.2)   KC   12-5 (10.1)   LAC  11-6 ( 8.8)   LV   1-16 ( 4.6)
NFC East    PHI  11-6 ( 8.2)   DAL   9-8 ( 9.6)   NYG   8-9 ( 8.9)   WAS  2-15 ( 7.7)
NFC North   DET  12-5 ( 9.7)   MIN  11-6 ( 8.4)   CHI  10-7 ( 9.0)   GB   10-7 ( 8.4)
NFC South   TB    9-8 ( 8.7)   ATL  7-10 ( 8.2)   NO   5-12 ( 7.5)   CAR  1-16 ( 7.1)
NFC West    SEA  15-2 ( 9.8)   LAR  14-3 (10.6)   SF    8-9 ( 9.1)   ARI  2-15 ( 6.5)


2026 playoffs, predicted -- the better seed hosts every round

AFC seeds: 1.HOU  2.NE  3.DEN  4.CIN  5.JAX  6.KC  7.LAC

AFC wild card   (HOU on a bye)
  (7) LAC at (2) NE    ->  NE   77.0%
  (6) K

# Random forest

The same protocol, the estimator swapped: five hundred trees instead of one,
each grown on its own bootstrap sample of the seasons and offered only a random
subset of the columns at every split, and the answer is their average.

Bagging is the family's first answer to the single tree's defining problem —
change a handful of games and the root split moves, taking every branch with it.
Averaging decorrelated trees cancels that wobble out. What this block measures is
how much it is worth on four thousand games with six to ten features: the tree's
numbers are printed next to the forest's, on the same 2025 holdout.

In [3]:
from functools import partial

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from nfl_trees.features import feature_names

# `n_estimators` is deliberately not in the search space below. Adding trees to
# a forest cannot overfit it: each one is fitted on its own bootstrap sample and
# the answer is their average, so more trees only makes that average steadier.
# It is a compute setting, not a regularizer, and the only question is where it
# stops paying. Measured on these folds with a mid-space configuration:
#
#     100 trees  cv auc 0.6304      300 trees  cv auc 0.6319
#     200 trees  cv auc 0.6303      500 trees  cv auc 0.6320
#
# Three hundred is the plateau -- five hundred buys the fourth decimal and costs
# 70% more. `n_jobs=1` because the parallelism is spent on the twelve folds
# instead, which is the better place for it (see `over_folds`).
FOREST = partial(RandomForestClassifier, n_estimators=300, n_jobs=1)

# Forty trials, where the tree got eighty. A trial here costs seconds instead of
# milliseconds, and bagging is the family that is least sensitive to its knobs.
# How insensitive is not something to take on faith: the spread printed below is
# the check on this number.
FOREST_TRIALS = 40


def forest_space(trial):
    """Six knobs: the tree's four that still mean something, plus the two bagging brings.

    `min_samples_leaf` starts at 1 here, where the tree's search had to start at
    10. A leaf holding three games still returns a probability of 0 or 1, but
    three hundred of those are averaged before anyone sees the number. The theory
    says to grow the trees deep and let the averaging do the regularizing; the
    folds get to check it rather than being told.

    The two knobs a single tree has no use for:

    - `max_features` is why bagging works at all. Each split only gets to look
      at a random subset of the columns, so the trees cannot all open with the
      same question; the lower it goes the more decorrelated they are, at the
      price of a worse tree individually. Over six to ten columns, 0.1 means one
      column offered per split and 1.0 means bagging with no feature sampling.
    - `max_samples` is how large each bootstrap sample is.

    `ccp_alpha` is gone: pruning every tree back is regularizing the variance
    that the averaging is already there to kill. `bootstrap` stays on -- turning
    it off, and drawing the split thresholds at random instead, is the next
    model in the catalog rather than a knob of this one.
    """
    return dict(
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        max_depth=trial.suggest_int("max_depth", 2, 24),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 250, log=True),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 200, log=True),
        max_features=trial.suggest_float("max_features", 0.1, 1.0),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
    )


# -- 1. tune on every candidate feature -------------------------------------- #
params_full, cv_full, study = tune(
    FOREST, forest_space, FULL, backend="processes", n_trials=FOREST_TRIALS
)
print(f"TPE search, {FOREST_TRIALS} trials over {len(TUNING_SEASONS)} folds (2013-2024)")
print("params  : " + "  ".join(f"{k}={v}" for k, v in params_full.items()))
print(f"cv auc  : {cv_full:.4f}   on all {len(FULL.columns)} features")

# What the search was worth: if the tenth-best trial is already this close, the
# forty-first would not have been the one to change the model.
ranked = sorted((t.value for t in study.trials if t.value is not None), reverse=True)
print(f"spread  : {ranked[0] - ranked[9]:.4f} between the best trial and the tenth, "
      f"{ranked[0] - ranked[-1]:.4f} to the worst")

# -- 2. ask each feature what it is worth ------------------------------------ #
forest_importance = permutation_table(FOREST, params_full, FULL, backend="processes")
print("\npermutation importance, out of fold: roc_auc lost when the column is shuffled")
print(forest_importance.round(5).to_string())

FOREST_KEEP, FOREST_DROPPED = selected(forest_importance)
print(f"\nkept    : {', '.join(FOREST_KEEP)}")
print(f"dropped : {', '.join(FOREST_DROPPED)}")

# -- 3. retune on the survivors ---------------------------------------------- #
FOREST_FEATURES = config_for(FOREST_KEEP)
FOREST_PARAMS, cv_reduced, _ = tune(
    FOREST,
    forest_space,
    FOREST_FEATURES,
    start_from=params_full,
    backend="processes",
    n_trials=FOREST_TRIALS,
)
cv_cut = cv_auc(FOREST, params_full, FOREST_FEATURES, "processes")
print(f"\nretuned on the {len(FOREST_KEEP)} survivors, builders now {FOREST_FEATURES.builders}")
print("params  : " + "  ".join(f"{k}={v}" for k, v in FOREST_PARAMS.items()))
print(f"cv auc  : {cv_reduced:.4f}   ({cv_reduced - cv_full:+.4f} against all {len(FULL.columns)})")
print(f"the cut : {cv_cut:.4f} with the step 1 params ({cv_cut - cv_full:+.4f}) -- what the")
print(f"          {len(FOREST_DROPPED)} dropped columns were worth to it")

# The tree never split on the columns it dropped, so cutting them cost it
# nothing. A forest cannot ignore a column that way: `max_features` offers a
# random subset at every split, and a weak column gets picked whenever the good
# ones are not on the table. Its share of the impurity decrease says how often.
grown = fit_model(FOREST, season.between(2011, 2024), params_full, FULL)
share = pd.Series(
    grown.named_steps["model"].feature_importances_,
    index=feature_names(grown.named_steps["prep"], FULL),
)
print(f"          the step 1 forest still spent {share[FOREST_DROPPED].sum():.0%} of its impurity")
print("          decrease on them")

# -- 4. the holdout neither step was allowed to see -------------------------- #
forest_report = holdout_report(
    FOREST,
    [
        (f"all {len(FULL.columns)}", params_full, FULL, cv_full),
        (f"kept {len(FOREST_KEEP)}", FOREST_PARAMS, FOREST_FEATURES, cv_reduced),
    ],
)
print(f"\ntrain 2011-2024 ({int(train.sum())} games), holdout {HOLDOUT} ({int(test.sum())} games)")
print(forest_report.to_string())

# -- 5. what the averaging bought -------------------------------------------- #
# One row each: the configuration its own block carries forward, not the best
# row either of them printed.
summary = pd.concat(
    {"decision tree": tree_report.tail(1), "random forest": forest_report.tail(1)}
).droplevel(1)
print("\nthe model each block carries into 2026, on the same holdout")
print(summary.to_string())

# The forest scores itself for free: each tree only saw its own bootstrap
# sample, so the rows it missed -- around a third of them -- are a test set for
# that tree, and `oob_decision_function_` averages every row over the trees that
# missed it. It is not the number to trust here, and it is worth seeing why: it
# pools the seasons, so a 2011 game is scored by trees that were grown on 2024.
# The rolling folds exist precisely to forbid that, and they come out lower.
forest = fit_model(
    partial(FOREST, oob_score=True), season.between(2011, HOLDOUT), FOREST_PARAMS, FOREST_FEATURES
)
played = season.between(2011, HOLDOUT)
oob_auc = roc_auc_score(y[played], forest.named_steps["model"].oob_decision_function_[:, 1])
print(f"\nrefit on 2011-{HOLDOUT} ({int(played.sum())} games) to predict 2026")
print(f"out-of-bag roc_auc {oob_auc:.4f}, against {cv_reduced:.4f} on the rolling folds")

TPE search, 40 trials over 12 folds (2013-2024)
params  : criterion=entropy  max_depth=17  min_samples_leaf=46  min_samples_split=167  max_features=0.46093878432770236  max_samples=0.5614658997297232
cv auc  : 0.6538   on all 10 features
spread  : 0.0025 between the best trial and the tenth, 0.0498 to the worst

permutation importance, out of fold: roc_auc lost when the column is shuffled
                           mean  std_err  seasons_up        t
away_pct_score_drive    0.04886  0.00629          12  7.77223
home_pct_score_drive    0.02581  0.00421          12  6.13276
home_pct_allowed_drive  0.01622  0.00560          10  2.89808
pct_home_win            0.01056  0.00385          10  2.74254
away_pct_allowed_drive  0.00592  0.00252           9  2.35136
pct_away_win            0.00363  0.00149           9  2.43042
day                     0.00003  0.00003           4  0.93381
playoff                 0.00000  0.00000           0      NaN
week                   -0.00039  0.00044          

In [4]:
# The same 272 games and the same bracket as the tree read, five hundred trees
# at a time. Where the two disagree is where the averaging changed somebody's
# season.
forest_standings = forecast_2026(forest, FOREST_FEATURES)

2026 regular season, 272 games predicted
W-L is the record the picks add up to, exp is the summed probabilities

AFC East    NE   14-3 (10.1)   BUF  13-4 ( 9.6)   MIA  5-12 ( 7.8)   NYJ  2-15 ( 6.4)
AFC North   PIT  10-7 ( 8.7)   BAL   8-9 ( 8.5)   CIN   8-9 ( 8.4)   CLE  3-14 ( 7.1)
AFC South   HOU  16-1 (10.3)   JAX  15-2 (10.2)   IND   9-8 ( 8.6)   TEN  2-15 ( 6.4)
AFC West    DEN  13-4 ( 9.5)   LAC   9-8 ( 8.9)   KC   7-10 ( 8.4)   LV   0-17 ( 5.9)
NFC East    PHI  10-7 ( 8.8)   DAL   8-9 ( 8.4)   NYG  6-11 ( 7.7)   WAS  3-14 ( 7.6)
NFC North   DET  12-5 ( 9.3)   CHI  10-7 ( 9.0)   MIN  10-7 ( 8.4)   GB    9-8 ( 8.5)
NFC South   TB   12-5 ( 9.1)   ATL   8-9 ( 8.5)   NO   4-13 ( 7.7)   CAR  1-16 ( 7.3)
NFC West    SEA  16-1 (10.7)   LAR  15-2 (10.0)   SF   12-5 ( 9.3)   ARI  2-15 ( 6.9)


2026 playoffs, predicted -- the better seed hosts every round

AFC seeds: 1.HOU  2.NE  3.DEN  4.PIT  5.JAX  6.BUF  7.LAC

AFC wild card   (HOU on a bye)
  (7) LAC at (2) NE    ->  NE   61.2%
  (6) 